# 19 — Comparação final de modelos (ELG)

Notebook de figura: lê `metrics.csv` + `predictions.npz` de cada modelo já treinado e
monta a tabela comparativa + scatter + histograma de Δz. Modelos ausentes são pulados.

**Convenção (Conclusões 9 e 14):** métrica principal = **σ_NMAD** + **η** (catástrofes) + **bias**.
RMSE/R² entram só como sanity check (não-robustos). Tabela consolidada completa em `docs/TABELA_RESULTADOS.md`.

In [ ]:
import sys
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt

ROOT = Path.cwd()
while not (ROOT / 'config.py').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from config import RESULTS_DIR
from src.evaluation.style import set_science_style
set_science_style()  # SciencePlots em todas as figuras (regra do projeto)

OBJ = 'ELG'
C_KMS = 299792.458
COMP_DIR = RESULTS_DIR / OBJ / 'comparison'
COMP_DIR.mkdir(parents=True, exist_ok=True)

# nome -> pasta (estrutura ATUAL: cada modelo salva metrics.csv + predictions.npz)
MODELS = {
    'XGBoost cru':  RESULTS_DIR / OBJ / 'xgboost_baseline',
    'CNN baseline': RESULTS_DIR / OBJ / 'cnn_baseline',
    'CNN linedet':  RESULTS_DIR / OBJ / 'cnn_linedet',
}
print('Objeto:', OBJ)

In [ ]:
def load_model(name, folder):
    mpath, ppath = folder / 'metrics.csv', folder / 'predictions.npz'
    if not mpath.exists() or not ppath.exists():
        print(f'  [SKIP] {name}: nao encontrado em {folder}')
        return None
    m = pd.read_csv(mpath).iloc[0].to_dict()
    p = np.load(ppath)
    print(f'  [OK]   {name}: {len(p["y_test"]):,} predicoes')
    return {'name': name, 'metrics': m, 'y_test': p['y_test'],
            'y_pred': p['y_pred'], 'delta_z': p['delta_z']}

loaded = {n: r for n, f in MODELS.items() if (r := load_model(n, f)) is not None}
if not loaded:
    raise RuntimeError('Nenhum modelo encontrado — treine os modelos antes.')

In [ ]:
# Tabela: metrica principal (sigma_NMAD + eta + bias) | sanity (RMSE/R2)
rows = []
for name, r in loaded.items():
    m = r['metrics']
    rows.append({'Modelo': name,
                 'σ_NMAD': m['nmad'], 'km/s': m['nmad'] * C_KMS,
                 'η>0.05 %': m.get('outliers_0.05_pct', np.nan),
                 'η>0.15 %': m.get('outliers_0.15_pct', np.nan),
                 'bias': m['bias'], 'RMSE': m['rmse'], 'R²': m['r2']})
df = pd.DataFrame(rows).set_index('Modelo')
df.to_csv(COMP_DIR / 'metrics_comparison.csv')
df.style.format({'σ_NMAD': '{:.5f}', 'km/s': '{:.0f}', 'η>0.05 %': '{:.2f}',
                 'η>0.15 %': '{:.3f}', 'bias': '{:+.4f}', 'RMSE': '{:.4f}', 'R²': '{:.3f}'})

In [ ]:
# Scatter predito x real (lado a lado)
n = len(loaded)
fig, axes = plt.subplots(1, n, figsize=(3.2*n, 3), squeeze=False)
for i, (name, r) in enumerate(loaded.items()):
    ax = axes[0, i]
    ax.scatter(r['y_test'], r['y_pred'], c=r['y_test'], cmap='viridis', alpha=0.4, s=6)
    lo, hi = r['y_test'].min(), r['y_test'].max()
    ax.plot([lo, hi], [lo, hi], 'r--', lw=1.2)
    ax.set_xlabel(r'$z_{\mathrm{true}}$'); ax.set_ylabel(r'$z_{\mathrm{pred}}$')
    ax.set_title(f"{name}\n$\\sigma_{{\\mathrm{{NMAD}}}}$={r['metrics']['nmad']:.5f} (~{r['metrics']['nmad']*C_KMS:.0f} km/s)")
fig.tight_layout()
fig.savefig(COMP_DIR / 'scatter_comparison.png')
plt.show()

In [ ]:
# Histograma de Δz/(1+z) sobreposto (zoom no nucleo)
fig, ax = plt.subplots(figsize=(8, 5))
for name, r in loaded.items():
    ax.hist(r['delta_z'], bins=200, range=(-0.02, 0.02), histtype='step', lw=1.8, label=name, density=True)
ax.axvline(0, color='k', ls='--', lw=1)
ax.set_xlabel('Δz / (1+z)'); ax.set_ylabel('densidade')
ax.set_title(f'Distribuição do resíduo — {OBJ} (zoom no núcleo)'); ax.legend()
plt.tight_layout()
plt.savefig(COMP_DIR / 'delta_z_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

**Nota:** o linedet tem σ_NMAD minúsculo mas cauda de catástrofes (η>0.15) — ver a rejeição por
confiança do heatmap em [10_cnn_interpretability.ipynb](10_cnn_interpretability.ipynb) e Tabela 4 de `docs/TABELA_RESULTADOS.md`.
Barras de erro (±) virão do benchmark multi-seed (`scripts/analysis/benchmark_summary.py`).